In [0]:
%sql
use catalog ecommerce

In [0]:
from pyspark.sql.functions import *
import json

In [0]:
for job in (10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009):
    with open(f"/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/{job}.json", "r") as job:
        job_details = json.load(job)
        job_details["partition"] = spark.range(1).select(date_format(current_timestamp(), "yyyyMMddHHmmssSSS").cast("bigint").alias('partition')).collect()[0]['partition']
        dbutils.notebook.run("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/ETL/raw_to_bronze", 500, {"job_parameters" : json.dumps(job_details)})

In [0]:
%sql
create or replace view ecommerce.bronze.vw_sellers as (
  select 
    try_Cast(seller_zip_code_prefix as string) as seller_zip_code_prefix,
    try_Cast(seller_id as string) as seller_id,
    try_Cast(seller_city as string) as seller_city,
    try_Cast(seller_state as string) as seller_state,
    file_name,
    partition
  from bronze.sellers
)

In [0]:
%sql
select * from bronze.vw_sellers

In [0]:
%sql
create or replace view ecommerce.bronze.vw_customers as (
  select 
    try_Cast(customer_id as string) as customer_id,
    try_Cast(customer_unique_id as string) as customer_unique_id,
    try_Cast(customer_zip_code_prefix as string) as customer_zip_code_prefix,
    try_Cast(customer_city as string) as customer_city,
    try_Cast(customer_state as string) as customer_state,
    file_name,
    partition
  from bronze.customers
)

In [0]:
%sql
select * from ecommerce.bronze.vw_customers

In [0]:
%sql
select * from bronze.orders

In [0]:
%sql
create or replace view ecommerce.bronze.vw_orders as (
  select 
    try_Cast(order_id as string) as order_id,
    try_Cast(customer_id as string) as customer_id,
    try_Cast(order_status as string) as order_status,
    try_to_timestamp(order_purchase_timestamp, "yyyy-MM-dd HH:mm:ss") as order_purchase_timestamp,
    try_to_timestamp(order_approved_at, "yyyy-MM-dd HH:mm:ss") as order_approved_at,
    try_to_timestamp(order_delivered_carrier_date, "yyyy-MM-dd HH:mm:ss") as order_delivered_carrier_date,
    try_to_timestamp(order_delivered_customer_date, "yyyy-MM-dd HH:mm:ss") as order_delivered_customer_date,
    try_to_timestamp(order_estimated_delivery_date, "yyyy-MM-dd HH:mm:ss") as order_estimated_delivery_date,
    file_name,
    partition
  from bronze.orders
)

In [0]:
%sql
select * from  bronze.vw_orders

In [0]:
%sql
create or replace view ecommerce.bronze.vw_geolocation as (
  select 
    try_Cast(geolocation_zip_code_prefix as string) as geolocation_zip_code_prefix,
    try_Cast(geolocation_lat as double) as geolocation_lat,
    try_Cast(geolocation_lng as double) as geolocation_lng,
    try_Cast(geolocation_city as string) as geolocation_city,
    try_Cast(geolocation_state as string) as geolocation_state,
    file_name,
    partition
  from bronze.geolocation
)

In [0]:
%sql
select * from ecommerce.bronze.vw_geolocation

In [0]:
%sql
create or replace view ecommerce.bronze.vw_order_payments as (
  select 
    try_Cast(order_id as string) as order_id,
    try_Cast(payment_sequential as int) as payment_sequential,
    try_Cast(payment_type as string) as payment_type,
    try_Cast(payment_installments as int) as payment_installments,
    try_Cast(payment_value as double) as payment_value,
    file_name,
    partition
  from bronze.order_payments
)

In [0]:
%sql
select * from ecommerce.bronze.vw_order_payments

In [0]:
%sql
create or replace view ecommerce.bronze.vw_order_items as (
  select 
    try_Cast(order_id as string) as order_id,
    try_Cast(order_item_id as int) as order_item_id,
    try_Cast(product_id as string) as product_id,
    try_Cast(seller_id as string) as seller_id,
    try_to_timestamp(shipping_limit_date, "yyyy-MM-dd HH:mm:ss") as shipping_limit_date,
    try_Cast(price as double) as price,
    try_Cast(freight_value as double) as freight_value,
    file_name,
    partition
  from bronze.order_items
)

In [0]:
%sql
select * from ecommerce.bronze.vw_order_items

In [0]:
%sql
create or replace view ecommerce.bronze.vw_product_category_name_translation as (
  select 
    try_Cast(product_category_name as string) as product_category_name,
    try_Cast(product_category_name_english as string) as product_category_name_english,
    file_name,
    partition
  from bronze.product_category_name_translation
)

In [0]:
%sql
select * from ecommerce.bronze.vw_product_category_name_translation

In [0]:
%sql
create or replace view ecommerce.bronze.vw_order_reviews as (
  select 
    try_Cast(review_id as string) as review_id,
    try_Cast(order_id as string) as order_id,
    try_Cast(review_score as int) as review_score,
    try_Cast(review_comment_title as string) as review_comment_title,
    try_Cast(review_comment_message as string) as review_comment_message,
    try_to_timestamp(review_creation_date, "yyyy-MM-dd HH:mm:ss") as review_creation_date,
    try_to_timestamp(review_answer_timestamp, "yyyy-MM-dd HH:mm:ss") as review_answer_timestamp,
    file_name,
    partition
  from ecommerce.bronze.order_reviews
)

In [0]:
%sql
select * from ecommerce.bronze.vw_order_reviews

In [0]:
%sql
create or replace view ecommerce.bronze.vw_products as (
  select 
    try_Cast(product_id as string) as product_id,
    try_Cast(product_category_name as string) as product_category_name,
    try_Cast(product_name_length as int) as product_name_length,
    try_Cast(product_description_length as int) as product_description_length,
    try_Cast(product_photos_qty as int) as product_photos_qty,
    try_Cast(product_weight_g as int) as product_weight_g,
    try_Cast(product_length_cm as int) as product_length_cm,
    try_Cast(product_height_cm as int) as product_height_cm,
    try_Cast(product_width_cm as int) as product_width_cm,
    file_name,
    partition
  from ecommerce.bronze.products
)

In [0]:
%sql
select * from ecommerce.bronze.vw_products